## CODE GENERATOR USING GRADIO UI

In [1]:
!uv pip install --quiet openai

In [2]:
# imports

import os
import subprocess
from rich import print
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [3]:
load_dotenv(override=True)

# Load groq envs
groq_api_key = os.getenv("GROQ_API_KEY")
groq_model = os.getenv("GROQ_MODEL") or "openai/gpt-oss-20b"
groq_base_url = os.getenv("GROQ_BASE_URL")

# Load ollama envs
ollama_model = os.getenv("OLLAMA_MODEL"),
ollama_base_url = os.getenv("OLLAMA_BASE_URL")
ollama_api_key = os.getenv("OLLAMA_API_KEY")

# Load openrouter envs
openrouter_model = os.getenv("OPENROUTER_MODEL")
openrouter_base_url = os.getenv("OPENROUTER_BASE_URL")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if groq_api_key:
    print(f"GROQ API Key exists and begins {groq_api_key[:8]}")
else:
    print("GROQ API Key not set")

GROQ API Key exists and begins gsk_GIUx

In [4]:
# Connect to groq client
groq = OpenAI(base_url=groq_base_url, api_key=groq_api_key)

# Connect to ollama client
ollama = OpenAI(base_url=ollama_base_url, api_key=ollama_api_key)

# Connect to openrouter client
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_base_url)

In [5]:
!uv pip install --quiet system_info

In [6]:
from system_info import sysinfo

# 1. Check what functions are hidden inside the actual sub-module
print(dir(sysinfo))

print("System Info Module Version:", sysinfo.sysInfo.items())

[
    'RAM_Types',
    'SystemInfo',
    '__builtins__',
    '__cached__',
    '__doc__',
    '__file__',
    '__loader__',
    '__name__',
    '__package__',
    '__spec__',
    '__warningregistry__',
    'cpuinfo',
    'math',
    'os',
    'os_info',
    'platform',
    'psutil',
    'shutil',
    'sk',
    'subprocess',
    'sys',
    'sysInfo'
]

System Info Module Version: dict_items([('Processor', 'AMD Ryzen 7 250 w/ Radeon 780M Graphics'), ('CPU', 16), 
('Ip', '10.138.93.152'), ('OS Version', '10.0.26200'), ('Total Disk Space', '951 GB'), ('HD Size', '951 GB'), 
('Available Space', '620 GB'), ('HD_Type', None), ('Operating System', 'Microsoft Windows 11 Home Single 
Language'), ('Host Name', 'LAPTOP-NJTCAHH8'), ('CPU_Core', None), ('Manufacturer', None), ('Model', None), 
('Ram_Type', None), ('Ram_Size', '15 GB'), ('Serial_Number', None)])

In [ ]:
models = ["gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "gemini-2.5-pro", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]

clients = {"openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

#### Overwrite this with the commands from yesterday

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

### And now, on with the main task

In [ ]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{sysinfo.sysInfo.items()}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [ ]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]


In [ ]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [ ]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

In [ ]:
compile_and_run()